# VAE loss ablation → sample $10^5$ → short evolve

Train small conditional set VAEs under **different auxiliary-loss assumptions**,
draw a **~$10^5$-particle** galaxy from each, evolve briefly, and compare
disk morphology (`A_2/A_0`, face-on maps, energy drift).

| Ablation | Auxiliaries |
|----------|-------------|
| `recon_only` | token MSE + CE + KL |
| `profiles` | + soft `Σ(R)`, `ρ(r)`, `⟨v_φ⟩` |
| `fourier` | + soft Fourier `m=1,2` |
| `maps` | + soft face-on / edge-on maps |
| `full` | profiles + Fourier + maps |

Defaults keep **training on CPU** and evolve with **CPU BH** (`bh` / `bh_c`)
so the GPU BH corpus can keep running. Bump `EPOCHS` / use a real manifest
for a serious comparison.

Docs: [`morton_generative.md`](../docs/morton_generative.md).

In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

# --- knobs (keep small while corpus occupies the GPU) ---
DEVICE = "cpu"
FORCE_METHOD = "bh"  # "bh_c" if compiled; avoid "gpu_bh" during corpus evolve
N_TRAIN = 64
BATCH_SIZE = 1
EPOCHS = 2
LR = 1e-3
SEED = 0
N_SAMPLE = 100_000  # ~1e5 particles for evolve
CHUNK = 256
EVOLVE_GYR = 0.2
DT = 0.02
MAP_PIX = 16

REPO = Path("..").resolve()
OUT = REPO / "notebooks/artifacts/vae_loss_ablation"
OUT.mkdir(parents=True, exist_ok=True)
MANIFEST_CANDIDATES = [
    REPO / "runs/mw_morton_corpus_v2/snapshot_manifest.json",
    REPO / "runs/mw_morton_corpus/snapshot_manifest.json",
]
print(f"torch {torch.__version__}  train_device={DEVICE}  force={FORCE_METHOD}")

In [ ]:
from galacticsics.ml.morton.dataset import MortonSnapshotDataset, collate_morton_batch
from galacticsics.ml.morton.index import write_snapshot_manifest
from galacticsics.ml.models.sequence_vae import SequenceVAE, SequenceVAEConfig


def synthetic_manifest(root: Path, n_runs: int = 6) -> Path:
    """Bar-flavoured synthetic snaps so Fourier/map ablations have signal."""
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(SEED)
    for i in range(n_runs):
        run = root / f"synth{i:04d}"
        run.mkdir(exist_ok=True)
        (run / "model.json").write_text(
            json.dumps(
                {
                    "label": f"synth{i}",
                    "disk": {
                        "mass": 12.0 + 2.0 * (i % 3),
                        "scale_length": 2.5,
                        "scale_height": 0.3,
                        "enabled": True,
                    },
                    "halo": {"v0": 3.5, "a": 30.0, "enabled": True},
                    "bulge": {"v0": 1.5, "a": 0.6, "enabled": True},
                    "disk_kinematics": {"toomre_q_target": 1.4, "sigma_r0": 0.5},
                }
            )
        )
        n = 800
        R = rng.exponential(3.0, n).clip(0.3, 14.0)
        phi = rng.uniform(0, 2 * np.pi, n)
        # mild m=2 so map/Fourier losses see structure
        bar = 0.25 * (i % 2)
        w = 1.0 + bar * np.cos(2 * phi)
        w /= w.sum()
        idx = rng.choice(n, size=n, replace=True, p=w)
        R, phi = R[idx], phi[idx]
        pos = np.stack(
            [R * np.cos(phi), R * np.sin(phi), rng.normal(0, 0.25, n)], axis=1
        )
        vel = np.stack(
            [-0.8 * np.sin(phi), 0.8 * np.cos(phi), rng.normal(0, 0.05, n)], axis=1
        )
        type_id = np.array([0] * 480 + [1] * 220 + [2] * 100, dtype=np.int32)
        np.savez(
            run / "ic_state.npz",
            pos=pos.astype(np.float64),
            vel=vel.astype(np.float64),
            mass=np.ones(n) / n,
            eps=np.full(n, 0.1),
            type_id=type_id,
        )
    return write_snapshot_manifest(root)


manifest = next((p for p in MANIFEST_CANDIDATES if p.is_file()), None)
if manifest is None:
    manifest = synthetic_manifest(OUT / "synthetic_corpus")
    print("no campaign manifest →", manifest)
else:
    print("using", manifest)

ds = MortonSnapshotDataset(manifest, n_particles=N_TRAIN, split="train", seed=SEED)
if len(ds) == 0:
    ds = MortonSnapshotDataset(manifest, n_particles=N_TRAIN, split=None, seed=SEED)
print(f"snapshots={len(ds)}  theta_dim={len(ds.theta_keys)}")

In [ ]:
def collate(batch):
    np_batch = collate_morton_batch(batch)
    return {k: torch.as_tensor(np_batch[k]) for k in ("c", "dm", "dx", "v", "theta")}


loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

# Zero auxiliaries → recon_only; turn subsets on for ablations
OFF = dict(lambda_sigma=0.0, lambda_rho=0.0, lambda_vphi=0.0, lambda_nonaxisym=0.0, lambda_maps=0.0)
PROFILES = dict(lambda_sigma=0.1, lambda_rho=0.1, lambda_vphi=0.1, lambda_nonaxisym=0.0, lambda_maps=0.0)
ABLATIONS = {
    "recon_only": OFF,
    "profiles": PROFILES,
    "fourier": {**PROFILES, "lambda_nonaxisym": 0.1},
    "maps": {**PROFILES, "lambda_maps": 0.1},
    "full": {**PROFILES, "lambda_nonaxisym": 0.1, "lambda_maps": 0.1},
}


def train_ablation(name: str, lambdas: dict) -> Path:
    cfg = SequenceVAEConfig(
        n_particles=N_TRAIN,
        theta_dim=len(ds.theta_keys),
        d_model=64,
        latent_dim=32,
        n_layers=1,
        n_heads=2,
        n_decode_layers=1,
        map_n_pix=MAP_PIX,
        **lambdas,
    )
    model = SequenceVAE(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    history = []
    for epoch in range(EPOCHS):
        losses = []
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(batch["c"], batch["dm"], batch["dx"], batch["v"], batch["theta"])
            metrics = model.loss(batch, out)
            opt.zero_grad(set_to_none=True)
            metrics["loss"].backward()
            opt.step()
            losses.append(float(metrics["loss"].detach().cpu()))
        history.append(float(np.mean(losses)))
        print(f"  {name}  epoch {epoch + 1}/{EPOCHS}  loss={history[-1]:.4f}")
    ckpt = OUT / f"vae_{name}.pt"
    torch.save(
        {
            "model": model.state_dict(),
            "config": asdict(cfg),
            "theta_keys": ds.theta_keys,
            "ablation": name,
            "history": history,
        },
        ckpt,
    )
    return ckpt


checkpoints = {}
for name, lambdas in ABLATIONS.items():
    print(f"=== train {name} ===")
    checkpoints[name] = train_ablation(name, lambdas)
print("checkpoints:", {k: str(v) for k, v in checkpoints.items()})

In [ ]:
from ntropy.analysis.disk_density import disk_azimuthal_fourier
from ntropy.config import ForceConfig, IntegratorConfig, ParallelConfig, RunConfig
from ntropy.particles import ParticleState
from ntropy.simulation import Simulation


def load_vae(ckpt_path: Path) -> tuple[SequenceVAE, list[str]]:
    blob = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    fields = SequenceVAEConfig.__dataclass_fields__
    raw = {k: v for k, v in blob["config"].items() if k in fields}
    if "fourier_modes" in raw:
        raw["fourier_modes"] = tuple(raw["fourier_modes"])
    cfg = SequenceVAEConfig(**raw)
    model = SequenceVAE(cfg).to(DEVICE)
    model.load_state_dict(blob["model"])
    model.eval()
    return model, list(blob["theta_keys"])


def sample_galaxy(model: SequenceVAE, theta: torch.Tensor, n: int) -> dict[str, np.ndarray]:
    with torch.no_grad():
        tok = model.generate(theta, n=n, chunk_size=CHUNK)
    pos = tok["dx"][0].astype(np.float64)
    vel = tok["v"][0].astype(np.float64)
    cid = tok["c"][0].astype(np.int32)
    # Prefer disk particles for the morphology test; top up from other comps if needed
    disk = np.flatnonzero(cid == 0)
    if disk.size >= int(0.5 * n):
        # keep all particles (live halo/bulge help the potential); report disk fraction
        pass
    mass = np.full(n, 1.0 / n, dtype=np.float64)
    eps = np.full(n, 0.08, dtype=np.float64)
    return {"pos": pos, "vel": vel, "mass": mass, "eps": eps, "type_id": cid}


def a2_median(pos: np.ndarray, mass: np.ndarray, type_id: np.ndarray) -> float:
    disk = type_id == 0
    if disk.sum() < 200:
        disk = np.ones(len(type_id), dtype=bool)
    out = disk_azimuthal_fourier(
        pos[disk], mass[disk], m=2, n_bins=10, r_max=12.0, z_max=0.5, min_count=30
    )
    return float(out["a_m_over_a0_median"])


def evolve_short(parts: dict[str, np.ndarray], end_gyr: float) -> tuple[ParticleState, list[float]]:
    state = ParticleState.from_arrays(
        parts["pos"],
        parts["vel"],
        parts["mass"],
        parts["eps"],
        type_id=parts["type_id"],
    )
    cfg = RunConfig()
    cfg.integrator = IntegratorConfig(type="leapfrog", dt=DT, end_time_gyr=float(end_gyr))
    cfg.force = ForceConfig(method=FORCE_METHOD, theta=0.7)
    cfg.parallel = ParallelConfig(enabled=False)
    cfg.output.write_final = False
    cfg.output.every = 0
    result = Simulation(cfg, state=state.copy()).run()
    energies = [float(e) for e in result.energies]
    return result.state, energies


# Shared conditioning: mean θ over the training set (or first item)
theta_np = np.mean([ds[i]["theta"] for i in range(min(len(ds), 32))], axis=0)
theta = torch.as_tensor(theta_np[None, :], dtype=torch.float32, device=DEVICE)

results = {}
for name, ckpt in checkpoints.items():
    print(f"=== sample+evolve {name}  N={N_SAMPLE} ===")
    model, _keys = load_vae(ckpt)
    parts = sample_galaxy(model, theta, N_SAMPLE)
    a2_ic = a2_median(parts["pos"], parts["mass"], parts["type_id"])
    disk_frac = float(np.mean(parts["type_id"] == 0))
    state_f, energies = evolve_short(parts, EVOLVE_GYR)
    a2_f = a2_median(state_f.pos, state_f.mass, state_f.type_id)
    e0, e1 = energies[0], energies[-1]
    dE = (e1 - e0) / abs(e0) if e0 != 0 else float("nan")
    results[name] = {
        "parts_ic": parts,
        "pos_f": np.asarray(state_f.pos),
        "type_id": np.asarray(state_f.type_id),
        "a2_ic": a2_ic,
        "a2_final": a2_f,
        "disk_frac": disk_frac,
        "dE_over_E": dE,
        "energies": energies,
    }
    np.savez_compressed(
        OUT / f"sample_{name}.npz",
        pos=parts["pos"],
        vel=parts["vel"],
        mass=parts["mass"],
        eps=parts["eps"],
        type_id=parts["type_id"],
        pos_final=results[name]["pos_f"],
    )
    print(
        f"  disk_frac={disk_frac:.2f}  A2/A0 ic={a2_ic:.3f} → final={a2_f:.3f}  "
        f"ΔE/E={dE:.3e}"
    )

In [ ]:
names = list(results.keys())
fig, axes = plt.subplots(2, len(names), figsize=(3.2 * len(names), 6.2), squeeze=False)

for j, name in enumerate(names):
    r = results[name]
    pos0 = r["parts_ic"]["pos"]
    tid = r["parts_ic"]["type_id"]
    disk = tid == 0
    if disk.sum() < 100:
        disk = np.ones(len(tid), dtype=bool)
    # subsample for plotting
    rng = np.random.default_rng(0)
    show = np.flatnonzero(disk)
    if show.size > 8000:
        show = rng.choice(show, size=8000, replace=False)
    axes[0, j].scatter(pos0[show, 0], pos0[show, 1], s=0.3, alpha=0.35, c="C0")
    axes[0, j].set_title(f"{name}\nA2={r['a2_ic']:.3f}")
    axes[0, j].set_aspect("equal")
    axes[0, j].set_xlim(-12, 12)
    axes[0, j].set_ylim(-12, 12)
    if j == 0:
        axes[0, j].set_ylabel("IC  y [kpc]")

    posf = r["pos_f"]
    tidf = r["type_id"]
    diskf = tidf == 0
    if diskf.sum() < 100:
        diskf = np.ones(len(tidf), dtype=bool)
    showf = np.flatnonzero(diskf)
    if showf.size > 8000:
        showf = rng.choice(showf, size=8000, replace=False)
    axes[1, j].scatter(posf[showf, 0], posf[showf, 1], s=0.3, alpha=0.35, c="C1")
    axes[1, j].set_title(f"A2={r['a2_final']:.3f}  ΔE/E={r['dE_over_E']:.1e}")
    axes[1, j].set_aspect("equal")
    axes[1, j].set_xlim(-12, 12)
    axes[1, j].set_ylim(-12, 12)
    axes[1, j].set_xlabel("x [kpc]")
    if j == 0:
        axes[1, j].set_ylabel(f"t={EVOLVE_GYR} Gyr  y [kpc]")

fig.suptitle("VAE loss ablation: disk face-on (IC → short evolve)", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "ablation_faceon.png", dpi=140, bbox_inches="tight")
plt.show()

# Summary table
print(f"{'ablation':12s}  {'disk%':>6s}  {'A2_ic':>7s}  {'A2_f':>7s}  {'ΔE/E':>10s}")
for name in names:
    r = results[name]
    print(
        f"{name:12s}  {100 * r['disk_frac']:5.1f}%  {r['a2_ic']:7.3f}  "
        f"{r['a2_final']:7.3f}  {r['dE_over_E']:10.3e}"
    )

## How to read this

- **`recon_only`** often yields a fuzzy / axisymmetrised cloud: token MSE does not
  force permutation-invariant morphology.
- **`profiles`** tightens radial structure but can still erase bars.
- **`fourier` / `maps` / `full`** should retain more non-axisymmetric power at IC;
  the short evolve checks whether that structure is dynamically plausible
  (not just a painted density).

Increase `EPOCHS`, point at `mw_morton_corpus_v2` dumps, and raise `EVOLVE_GYR`
once the corpus GPU job is idle. Artifacts: `notebooks/artifacts/vae_loss_ablation/`.